# PySpark reference 2: DataFrame API

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-reference-02-dataframe-api.ipynb)

Creating DataFrames and schemas, selecting and computing columns, filtering and deduplicating, sorting and paging, unions and set operations, composing steps with `transform`, sampling, inspecting and collecting results. Practise it in [PySpark practice 2: DataFrame API](pyspark-practice-02-dataframe-api.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0 pandas pyarrow

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and small helpers: `plan_lines(df)` returns what `explain()` prints, `final_plan(df)` runs a query and returns the plan Spark actually used, and `expected_error()` prints the error class of an example that fails on purpose.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        print(type(error).__name__ + ":", str(error).split(".")[0][:100])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Contents

| section | entries |
| --- | --- |
| [Creating DataFrames](#creating-dataframes) | [`spark.createDataFrame(data, schema=None)`](#sparkcreatedataframedata-schemanone), [Schemas: `StructType`, `StructField` and DDL strings](#schemas-structtype-structfield-and-ddl-strings), [`spark.range(start, end=None, step=1, numPartitions=None)`](#sparkrangestart-endnone-step1-numpartitionsnone) |
| [Selecting and computing columns](#selecting-and-computing-columns) | [`select(*cols)` and `selectExpr(*expr)`](#selectcols-and-selectexprexpr), [`withColumn(colName, col)` and `withColumns(colsMap)`](#withcolumncolname-col-and-withcolumnscolsmap), [Renaming: `withColumnRenamed`, `withColumnsRenamed`, `toDF`, `alias`](#renaming-withcolumnrenamed-withcolumnsrenamed-todf-alias), [`drop(*cols)`](#dropcols), [Referring to columns](#referring-to-columns) |
| [Filtering and deduplicating rows](#filtering-and-deduplicating-rows) | [`where(condition)` / `filter(condition)`](#wherecondition--filtercondition), [`distinct()` and `dropDuplicates(subset=None)`](#distinct-and-dropduplicatessubsetnone) |
| [Sorting and limiting](#sorting-and-limiting) | [`orderBy(*cols, ascending=True)` / `sort(...)`](#orderbycols-ascendingtrue--sort), [`limit(n)`, `offset(n)` and `tail(n)`](#limitn-offsetn-and-tailn) |
| [Combining DataFrames](#combining-dataframes) | [`union`, `unionAll` and `unionByName`](#union-unionall-and-unionbyname), [`intersect`, `subtract` and `exceptAll`](#intersect-subtract-and-exceptall) |
| [Composing transformations](#composing-transformations) | [`DataFrame.transform(func, *args, **kwargs)`](#dataframetransformfunc-args-kwargs) |
| [Sampling and splitting](#sampling-and-splitting) | [`sample`, `sampleBy` and `randomSplit`](#sample-sampleby-and-randomsplit) |
| [Inspecting data](#inspecting-data) | [`show(n=20, truncate=True, vertical=False)`](#shown20-truncatetrue-verticalfalse), [`describe(*cols)` and `summary(*statistics)`](#describecols-and-summarystatistics), [`count`, `isEmpty`, `first`, `head`, `take`](#count-isempty-first-head-take) |
| [Moving data to the driver](#moving-data-to-the-driver) | [`collect`, `toPandas`, `toLocalIterator` and `Row`](#collect-topandas-tolocaliterator-and-row) |

## Creating DataFrames

### `spark.createDataFrame(data, schema=None)`

Builds a DataFrame from local data on the driver. Use it for tests, small lookup tables and examples; real data comes
from `spark.read` ([PySpark reference 10: reading and writing data](pyspark-reference-10-reading-and-writing.ipynb)) or tables.

| parameter | type | description |
| --- | --- | --- |
| `data` | list of tuples, `Row`s or dicts; a pandas DataFrame; an RDD | the rows |
| `schema` | DDL string, `StructType`, list of column names, or None | column names and types. A DDL string such as `"id int, name string"` is the shortest exact form; a list of names infers types; None infers names and types from `Row`s or dicts |
| `samplingRatio` | float | fraction of rows sampled when inferring types from an RDD |
| `verifySchema` | bool, default True | check every value against the schema |

**Returns:** a DataFrame.

**Notes**
- Always pass an explicit schema in pipelines and tests. Inference picks `long` for Python ints and `double` for
  floats, fails on columns that are all `None`, and differs between pandas and Python input.
- Python `Decimal` values need a `decimal(p,s)` type, `datetime` becomes `timestamp`, `date` becomes `date`.

In [ ]:
from pyspark.sql import Row

ddl = spark.createDataFrame([(1, "a", 2.5)], "id int, code string, score double")
inferred = spark.createDataFrame([(1, "a", 2.5)], ["id", "code", "score"])
rows = spark.createDataFrame([Row(id=1, code="a"), Row(id=2, code=None)])
print(ddl.dtypes, inferred.dtypes, rows.dtypes, sep="\n")

### Schemas: `StructType`, `StructField` and DDL strings

A schema is a `StructType`: an ordered list of `StructField(name, dataType, nullable=True, metadata=None)`. Nested
types are `ArrayType(elementType)`, `MapType(keyType, valueType)` and `StructType` again. The same schema can be
written as a DDL string.

| inspect | returns |
| --- | --- |
| `df.schema` | the `StructType` |
| `df.printSchema()` | a tree view |
| `df.columns` | list of column names |
| `df.dtypes` | list of `(name, type string)` |
| `df.schema.simpleString()` / `.json()` | compact string / JSON, useful for storing a contract |
| `StructType.fromJson(json.loads(s))` / `_parse_datatype_string(ddl)` | rebuild a schema |

**Notes:** `nullable=False` is a hint to the optimizer, not a constraint Spark enforces on every source; validate
required fields yourself.

In [ ]:
from pyspark.sql.types import ArrayType, DecimalType, IntegerType, StringType, StructField, StructType

schema = StructType([
    StructField("order_id", IntegerType(), nullable=False),
    StructField("customer", StructType([StructField("id", IntegerType()), StructField("name", StringType())])),
    StructField("tags", ArrayType(StringType())),
    StructField("amount", DecimalType(12, 2)),
])
nested = spark.createDataFrame([(1, (7, "Gamma Sports"), ["new"], D("10.00"))], schema)
nested.printSchema()
print(nested.schema.simpleString())
same = spark.createDataFrame([(1, (7, "Gamma Sports"), ["new"], D("10.00"))],
                             "order_id int, customer struct<id:int,name:string>, tags array<string>, amount decimal(12,2)")
print("DDL string gives the same types:", same.schema.simpleString() == nested.schema.simpleString())

### `spark.range(start, end=None, step=1, numPartitions=None)`

A single `long` column named `id`, generated in parallel without any driver data. With one argument it counts from 0.

**Use it for:** generating test data at scale, calendars (`date_add(start, id)`), and scaffolds for cross joins.

In [ ]:
spark.range(3).show()
spark.range(0, 10, 3).select("id", F.date_add(F.lit("2026-01-01").cast("date"), F.col("id").cast("int")).alias("day")).show()

## Selecting and computing columns

### `select(*cols)` and `selectExpr(*expr)`

`select` returns a DataFrame with exactly the given columns or expressions, in that order. Arguments can be column
names, `Column` expressions, or `"*"`. `selectExpr` takes SQL expression strings, including aliases (`"amount * 2 AS
doubled"`).

**Returns:** a DataFrame.

**Notes:** a `select` with an explicit column list at the end of each step documents the schema the step produces and
stops new upstream columns from leaking downstream.

In [ ]:
orders.select("order_id", F.col("amount"), (F.col("amount") * 1.2).alias("gross")).show(3)
orders.selectExpr("order_id", "amount * 1.2 AS gross", "upper(status) AS status").show(3)
orders.select("*", F.lit("EUR").alias("currency")).show(2)

### `withColumn(colName, col)` and `withColumns(colsMap)`

`withColumn` adds a column, or replaces it if the name exists, and keeps all others. `withColumns` takes a dict and
applies several at once.

**Returns:** a DataFrame.

**Notes:** each `withColumn` adds a projection to the plan; calling it in a long Python loop (hundreds of columns)
makes planning slow. Build the columns in one `select` or `withColumns` instead. The expressions in one
`withColumns` call cannot refer to each other.

In [ ]:
orders.withColumn("amount", F.col("amount") * 2).select("order_id", "amount").show(2)
orders.withColumns({
    "order_date": F.to_date("order_ts"),
    "is_large": F.col("amount") >= 200,
    "status": F.upper("status"),
}).select("order_id", "order_date", "is_large", "status").show(3)

### Renaming: `withColumnRenamed`, `withColumnsRenamed`, `toDF`, `alias`

| method | description |
| --- | --- |
| `df.withColumnRenamed(existing, new)` | rename one column (no error if `existing` is missing) |
| `df.withColumnsRenamed({old: new, ...})` | rename several |
| `df.toDF(*names)` | replace all names positionally |
| `F.col(c).alias(name)` | name an expression inside `select` |
| `df.alias(name)` | name the DataFrame, for qualifying columns in joins |

In [ ]:
raw = spark.createDataFrame([(1, "x")], "`Order ID` int, `Customer Name` string")
print(raw.toDF(*[c.lower().replace(" ", "_") for c in raw.columns]).columns)
print(orders.withColumnsRenamed({"order_ts": "ordered_at", "amount": "amount_eur"}).columns)

### `drop(*cols)`

Returns a DataFrame without the given columns. Names that do not exist are ignored silently, which makes typos easy
to miss.

**Notes:** to drop one side's copy of a column after an expression join, pass the `Column` (`df.drop(other.col)`),
not the name.

In [ ]:
print(orders.drop("order_ts", "not_a_column").columns)

### Referring to columns

| form | use when |
| --- | --- |
| `"name"` | the function accepts a name (`select`, `groupBy`, `orderBy`, `drop`, most `F` functions) |
| `F.col("name")` | building expressions: arithmetic, comparisons, methods like `.cast`, `.isNull`, `.alias` |
| `df.name` / `df["name"]` | picking a column of a specific DataFrame, for example one side of a join |
| ``F.col("`odd name`")`` | names with spaces, dots or other special characters (backticks) |
| `F.col("struct.field")` | a field inside a struct column |

**Notes:** a column literally named `a.b` and a struct field `b` of column `a` are written differently: ``F.col("`a.b`")``
vs `F.col("a.b")`.

In [ ]:
odd = spark.createDataFrame([(1, 2, (3,))], "`order.id` int, `unit price` int, s struct<field:int>")
odd.select(F.col("`order.id`"), F.col("`unit price`") * 2, F.col("s.field")).show()

## Filtering and deduplicating rows

### `where(condition)` / `filter(condition)`

Identical methods. The condition is a boolean `Column` or a SQL string. Rows where it is **true** are kept; rows where
it is false **or null** are dropped.

**Notes**
- Combine with `&`, `|`, `~` and parenthesize each comparison: `(F.col("a") > 1) & (F.col("b") == "x")`.
- `col != "x"` drops rows where `col` is null; add `| F.col(c).isNull()` if they should stay.
- Consecutive filters are merged by the optimizer; split them for readability freely.

In [ ]:
orders.where((F.col("status") == "paid") & (F.col("amount") > 100)).select("order_id", "amount").show()
orders.filter("status IN ('placed', 'returned') AND amount < 100").select("order_id", "status", "amount").show()
print("region != 'North':", customers.where(F.col("region") != "North").count(),
      "| including nulls:", customers.where((F.col("region") != "North") | F.col("region").isNull()).count())

### `distinct()` and `dropDuplicates(subset=None)`

`distinct()` removes rows that are identical in **every** column. `dropDuplicates(subset)` keeps one row per distinct
combination of the `subset` columns, and **which row it keeps is not defined**.

**Notes:** to keep a specific row per key (the latest, the largest), use `row_number()` over a window ordered by the
rule ([PySpark reference 8: window functions](pyspark-reference-08-window-functions.ipynb)) or `max_by`. Both methods shuffle. For streams,
`dropDuplicatesWithinWatermark` bounds the state kept.

In [ ]:
print("events:", events.count(), "| distinct:", events.distinct().count())
orders.select("customer_id", "status").distinct().groupBy("customer_id").count().orderBy("customer_id").show(3)
orders.dropDuplicates(["customer_id"]).select("customer_id", "order_id").orderBy("customer_id").show()

## Sorting and limiting

### `orderBy(*cols, ascending=True)` / `sort(...)`

A global sort: shuffles by range so the output is ordered across all partitions. Pass `F.desc("c")`,
`F.col("c").asc_nulls_last()` or `ascending=[True, False]` for mixed directions. Nulls sort first ascending and last
descending by default.

`sortWithinPartitions(*cols)` sorts each partition without a shuffle; use it before writing files so each file is
sorted (better compression and min/max statistics), when global order does not matter.

**Notes:** order is only guaranteed for the immediate result: a later join or aggregation discards it. Sort as the
last step, and add a unique tie-breaker when the order must be repeatable.

In [ ]:
orders.orderBy(F.desc("amount"), "order_id").select("order_id", "amount").show(3)
customers.orderBy(F.col("region").asc_nulls_last(), "customer_id").select("customer_id", "region").show()

### `limit(n)`, `offset(n)` and `tail(n)`

`limit(n)` keeps the first `n` rows; `offset(n)` skips `n` rows (Spark 3.4+); `tail(n)` returns the last `n` rows to the
driver as a list. Without an `orderBy` first, "first" means whichever rows Spark produces first.

**Use it for:** samples for inspection and pagination (`orderBy(key).offset(page * size).limit(size)`).

In [ ]:
orders.orderBy("order_id").offset(4).limit(3).select("order_id").show()
print([r.order_id for r in orders.orderBy("order_id").tail(2)])

## Combining DataFrames

### `union`, `unionAll` and `unionByName`

Stack the rows of two DataFrames. `union` and `unionAll` (the same thing) match columns **by position**;
`unionByName` matches them **by name**, and with `allowMissingColumns=True` fills columns missing on one side with
null. None of them remove duplicates; add `.distinct()` for SQL's `UNION`.

**Notes:** a positional `union` of two DataFrames whose columns are in a different order silently puts values in the
wrong columns when the types are compatible. Prefer `unionByName` for anything not built in the same function.

In [ ]:
feed_a = spark.createDataFrame([(1, "paid")], "order_id int, status string")
feed_b = spark.createDataFrame([("shipped", 2)], "status string, order_id int")
feed_c = spark.createDataFrame([(3, "placed", "web")], "order_id int, status string, channel string")
feed_a.unionByName(feed_b).unionByName(feed_c, allowMissingColumns=True).show()
with expected_error():
    feed_a.union(feed_b).show()  # columns are matched by position: order_id meets status and the cast fails

### `intersect`, `subtract` and `exceptAll`

Set operations on whole rows (all columns, compared by position):

| method | SQL | returns | duplicates |
| --- | --- | --- | --- |
| `a.intersect(b)` | `INTERSECT` | rows in both | removed |
| `a.intersectAll(b)` | `INTERSECT ALL` | rows in both | kept, by multiplicity |
| `a.subtract(b)` | `EXCEPT` | rows of `a` not in `b` | removed |
| `a.exceptAll(b)` | `EXCEPT ALL` | rows of `a` not in `b` | kept, by multiplicity |

**Use it for:** comparing two versions of a dataset in tests: `a.exceptAll(b)` and `b.exceptAll(a)` both empty means
the two are equal including duplicates.

In [ ]:
a = spark.createDataFrame([(1,), (1,), (2,), (3,)], "k int")
c = spark.createDataFrame([(1,), (3,), (4,)], "k int")
print("intersect:", sorted(r.k for r in a.intersect(c).collect()), "| intersectAll:", sorted(r.k for r in a.intersectAll(c).collect()))
print("subtract: ", sorted(r.k for r in a.subtract(c).collect()), "| exceptAll:", sorted(r.k for r in a.exceptAll(c).collect()))

## Composing transformations

### `DataFrame.transform(func, *args, **kwargs)`

Calls `func(df, *args, **kwargs)` and returns its result, so reusable steps written as plain functions chain like
built-in methods.

**Use it for:** pipelines built from small, individually tested functions (`clean`, `enrich`, `add_flags`).

In [ ]:
def only_settled(df):
    return df.where(F.col("status").isin("paid", "shipped"))

def with_vat(df, rate):
    return df.withColumn("gross", F.round(F.col("amount") * (1 + rate), 2))

orders.transform(only_settled).transform(with_vat, rate=0.2).select("order_id", "status", "gross").show(4)

## Sampling and splitting

### `sample`, `sampleBy` and `randomSplit`

| method | description |
| --- | --- |
| `df.sample(fraction, seed=None, withReplacement=False)` | each row kept with probability `fraction` (not an exact count) |
| `df.sampleBy(col, fractions, seed)` | stratified: a fraction per value of `col` |
| `df.randomSplit(weights, seed)` | list of DataFrames, e.g. `[0.8, 0.2]` for train/test |

**Notes:** results repeat only with the same seed **and** the same partitioning of the input. For an exact number of
rows use `orderBy(F.rand(seed)).limit(n)` (which sorts everything).

In [ ]:
big = spark.range(1000, numPartitions=4)
print("sample(0.1):", big.sample(0.1, seed=7).count(), "rows (about 100)")
train, test = big.randomSplit([0.8, 0.2], seed=7)
print("split:", train.count(), "+", test.count())
print("by status:", sorted((r.status, r.n) for r in orders.sampleBy("status", {"paid": 1.0, "shipped": 0.0}, seed=1).groupBy("status").agg(F.count("*").alias("n")).collect()))

## Inspecting data

### `show(n=20, truncate=True, vertical=False)`

Prints the first `n` rows as a table. `truncate=True` cuts values at 20 characters; pass `False` or a number. `vertical=True`
prints one block per row, useful for wide rows.

**Notes:** `show` is an action and runs the query; it only prints. Use `take(n)` or `collect()` to get rows back.

In [ ]:
orders.show(2, vertical=True)

### `describe(*cols)` and `summary(*statistics)`

Basic statistics as a DataFrame of **strings**. `describe` gives count, mean, stddev, min and max; `summary` also
gives quartiles by default and accepts a list such as `"count", "min", "25%", "75%", "max", "count_distinct"`.

**Notes:** they scan the whole input. Cast before comparing values, since every statistic is a string.

In [ ]:
orders.select("amount").summary("count", "min", "50%", "max").show()

### `count`, `isEmpty`, `first`, `head`, `take`

| method | returns |
| --- | --- |
| `count()` | number of rows (scans everything) |
| `isEmpty()` | True if there are no rows; stops at the first row, cheaper than `count() == 0` |
| `first()` / `head()` | the first `Row` or None |
| `head(n)` / `take(n)` | a list of the first `n` `Row`s |

In [ ]:
print(orders.count(), orders.where("amount > 5000").isEmpty(), orders.orderBy("order_id").first().order_id, len(orders.take(3)))

## Moving data to the driver

### `collect`, `toPandas`, `toLocalIterator` and `Row`

| method | returns | memory |
| --- | --- | --- |
| `collect()` | list of `Row` | every row on the driver at once |
| `toPandas()` | pandas DataFrame (Arrow-accelerated with `spark.sql.execution.arrow.pyspark.enabled`) | every row on the driver |
| `toLocalIterator()` | iterator of `Row`, one partition at a time | one partition |
| `toArrow()` | a PyArrow Table (Spark 4.0) | every row |

A `Row` behaves like a named tuple: `row.amount`, `row["amount"]`, `row[4]` and `row.asDict()`.

**Notes:** collecting a large DataFrame is the most common cause of driver out-of-memory errors. Aggregate or limit
first, and write large results to storage instead.

In [ ]:
row = orders.orderBy("order_id").first()
print(row.order_id, row["status"], row.asDict()["amount"])
print(orders.groupBy("status").count().orderBy("status").toPandas().to_dict("records"))
print(sum(1 for _ in orders.toLocalIterator()))

**Next:** practise these entries in [PySpark practice 2: DataFrame API](pyspark-practice-02-dataframe-api.ipynb).